# Rust CUDA kernels in `%%rust` cells

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xavierforge/colab-rust/blob/main/examples/04_cuda_oxide.ipynb)

Write a GPU kernel in Rust with [cuda-oxide](https://github.com/NVIDIA/cuda-rust/tree/main/cuda-oxide) (the SIMT kernel model in NVIDIA's cuda-rust), compile it inside a `%%rust` cell, and launch it on Colab's T4, without leaving the notebook.

**Needs a GPU runtime** (Runtime > Change runtime type). Setup takes a few minutes, and so does the first `%%rust` cell that pulls in cuda-oxide's crates; after that a kernel compiles in seconds. The Tesla T4 (sm_75), L4 (sm_89), A100 (sm_80) and RTX PRO 6000 Blackwell (sm_120a) are the measured cards; the setup script accepts any other Colab GPU with a warning and derives its PTX target from the compute capability. Colab's Ubuntu 24.04 image is the only one it accepts.

What the setup installs: a minimal CUDA 13.4 toolkit (740 MB instead of the multi-GB full metapackage), clang-21, the stable Rust that cuda-oxide pins (1.99.1 with the rustc-dev and llvm-tools components), an exact cuda-rust checkout, and a prebuilt codegen backend. All of it lives under `~/.cache/colab-rust`; the only file in `/content` is the `colab_rust.py` that the base setup installs.


## 1. Check the runtime


In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
!grep PRETTY_NAME /etc/os-release

## 2. Install

`setup-cuda-oxide.sh` runs `setup.sh` first (Rust, evcxr, the `%%rust` magic) if this session does not have it yet.


In [ ]:
!curl -fsSL -o /tmp/setup-cuda-oxide.sh https://raw.githubusercontent.com/xavierforge/colab-rust/main/setup-cuda-oxide.sh
!bash /tmp/setup-cuda-oxide.sh
%load_ext colab_rust

## 3. Point `%%rust` at the pinned toolchain and the backend

This is what `cargo oxide run` sets up behind the scenes. `RUSTFLAGS` is given in full because evcxr applies `:build_env` after its own flags, so this replaces them; `-Cprefer-dynamic` is evcxr's own and has to stay. `RUSTC_BOOTSTRAP=1` unlocks the `-Z` flags on the stable toolchain, the same way cuda-oxide's own `.cargo/config.toml` does. The three `:dep` lines pull in cuda-rust's crates from the checkout, `cuda-core` included, so that the cell and `cuda-host` share one copy of it; this is the one slow cell (a few minutes, once per session).

The PTX target is not in this cell: the setup script derived it from the GPU (`sm_75` on a T4, `sm_120a` on an RTX PRO 6000) and wrote it as `CUDA_OXIDE_TARGET` into `~/.cargo/config.toml`, whose `[env]` table reaches every rustc that cargo starts, this cell's build included. So the same cell works on every Colab GPU. The `env!` on the last line reads the value at compile time, which is where the backend reads it too, so a successful print is the proof that it arrived.

`:cache 2000` turns on evcxr's compile cache. In this session it changes nothing; it is what lets [05_cuda_oxide_drive_cache.ipynb](https://colab.research.google.com/github/xavierforge/colab-rust/blob/main/examples/05_cuda_oxide_drive_cache.ipynb) keep the compiled crates on your Drive so the next session skips most of this cell.


In [ ]:
%%rust
:toolchain 1.99.1
:opt 3
:cache 2000
:build_env RUSTC_BOOTSTRAP=1
:build_env CUDA_HOME=/usr/local/cuda-13
:build_env CUDA_TOOLKIT_PATH=/usr/local/cuda-13
:build_env CUDA_OXIDE_LLC=/opt/colab-rust/cuda-oxide/bin/llc
:build_env RUSTFLAGS=-Cprefer-dynamic -Zcodegen-backend=/opt/colab-rust/cuda-oxide/lib/librustc_codegen_cuda.so -Zmir-enable-passes=-JumpThreading -Zalways-encode-mir -Csymbol-mangling-version=v0 -Cdebug-assertions=off -Coverflow-checks=off
:dep cuda-core = { path = "/opt/colab-rust/cuda-oxide/src/cuda-core" }
:dep cuda-device = { path = "/opt/colab-rust/cuda-oxide/src/cuda-oxide/crates/cuda-device" }
:dep cuda-host = { path = "/opt/colab-rust/cuda-oxide/src/cuda-oxide/crates/cuda-host" }
println!("cuda-oxide configured for {}", env!("CUDA_OXIDE_TARGET", "no PTX target: run the setup cell first"));

## 4. A kernel, compiled and launched

`#[cuda_module] mod kernels` is compiled to PTX by the cuda-oxide backend and embedded in this cell's build. `kernels::load(&ctx)` reads it back from the cell's own shared object (the loader follows the module's artifact anchor, so it works inside evcxr's dynamically loaded cells) and hands you the typed launch API. The explicit `kernels::LoadedModule` type lets evcxr keep `loaded` for later cells.

In [4]:
%%rust
use cuda_core::LaunchConfig;
use cuda_core::{CudaContext, DeviceBuffer};
use cuda_device::{DisjointSlice, cuda_module, kernel, thread};

#[cuda_module]
mod kernels {
    use super::*;

    #[kernel]
    pub fn vecadd(a: &[f32], b: &[f32], mut c: DisjointSlice<f32>) {
        let idx = thread::index_1d();
        let raw = idx.get();
        if let Some(out) = c.get_mut(idx) {
            *out = a[raw] + b[raw];
        }
    }
}

let ctx = CudaContext::new(0).unwrap();
let stream = ctx.default_stream();
let loaded: kernels::LoadedModule = kernels::load(&ctx).unwrap();

const N: usize = 1024;
let a: Vec<f32> = (0..N).map(|i| i as f32).collect();
let b: Vec<f32> = (0..N).map(|i| (i * 2) as f32).collect();
let a_dev = DeviceBuffer::from_host(&stream, &a).unwrap();
let b_dev = DeviceBuffer::from_host(&stream, &b).unwrap();
let mut c_dev = DeviceBuffer::<f32>::zeroed(&stream, N).unwrap();
unsafe { loaded.vecadd(&stream, LaunchConfig::for_num_elems(N as u32), &a_dev, &b_dev, &mut c_dev) }.unwrap();
println!("{:?}", &c_dev.to_host_vec(&stream).unwrap()[..5]);

[0.0, 3.0, 6.0, 9.0, 12.0]


## 5. GPU state survives across cells

`ctx`, `stream`, the device buffers and `loaded` all live on in the evcxr runtime, so the next cell can launch again without re-uploading anything.


In [5]:
%%rust
let mut d_dev = DeviceBuffer::<f32>::zeroed(&stream, N).unwrap();
unsafe { loaded.vecadd(&stream, LaunchConfig::for_num_elems(N as u32), &a_dev, &c_dev, &mut d_dev) }.unwrap();
println!("{:?}", &d_dev.to_host_vec(&stream).unwrap()[..5]);

[0.0, 4.0, 8.0, 12.0, 16.0]


## Things to know

- **Redefining the module replaces it.** A later cell with its own `#[cuda_module] mod kernels { ... }` swaps in the new kernels; call `kernels::load` again in that cell to get the new `LoadedModule`.
- **`ThreadIndex` is not `Copy`.** Take `idx.get()` before `c.get_mut(idx)`, as above, or the compiler will complain about a moved value.
- **Annotate `loaded`.** evcxr can only keep a variable across cells when it knows the type name, so write `let loaded: kernels::LoadedModule = ...`. Inside a `{ }` block nothing needs annotating.
- **Stop clears variables.** Pressing stop interrupts the running cell but resets evcxr's variables, including `ctx` and the buffers; items such as `mod kernels` from earlier cells stay. Run the kernel cell (section 4) again to rebuild the state.
- **Back to plain Rust.** `%rust_reset` restarts evcxr on the default stable toolchain without the backend; run the configuration cell (section 3) again to return to cuda-oxide.
- **`DriverError(218, "a PTX JIT compilation failed")` from `kernels::load`** means the PTX was built for a newer GPU than this session has. The setup cell sets `CUDA_OXIDE_TARGET` for the session's GPU in `~/.cargo/config.toml`; if you overrode it, put it back, `%rust_reset`, and run the configuration cell again.
- **Other GPUs.** The setup script derives the target from the compute capability the way `cargo oxide run` does (`sm_89` on an L4, `sm_80` on an A100, `sm_120a` on consumer or workstation Blackwell) and writes it to `~/.cargo/config.toml`; the T4, L4, A100 and RTX PRO 6000 have been run end to end, the H100 not yet.
- **Coming back tomorrow.** The configuration cell's build can be kept on your Drive: [05_cuda_oxide_drive_cache.ipynb](https://colab.research.google.com/github/xavierforge/colab-rust/blob/main/examples/05_cuda_oxide_drive_cache.ipynb) restores it at the start of a session and saves it at the end. A `%%rust` cell containing just `:cache` prints the cache's entries, disk use and hits.
- **What is pinned.** cuda-oxide commit `cced2a35` (2026-11-02) of [NVIDIA/cuda-rust](https://github.com/NVIDIA/cuda-rust), Rust 1.99.1, CUDA 13.4; `cuda-core` comes from the same checkout. cuda-oxide is alpha software; the setup script pins these on purpose and `COLAB_CUDA_OXIDE_REF` overrides the commit. Commits from before cuda-oxide moved into cuda-rust (2026-09-30) are refused; colab-rust v0.2.0 covers those.
